# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a reproducible template for loading and exploring the FAIR² dataset using the `mlcroissant` library and pandas. All dataset components (record sets, fields, columns) are referenced by their Croissant `@id`, ensuring robust, schema-aware data handling.

### Dataset Source

FAIR² dataset Croissant schema:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure that the mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using [mlcroissant](https://github.com/mlcommons/croissant).

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Croissant dataset (schema + metadata)
dataset = mlc.Dataset(croissant_url)

# Access and pretty-print dataset metadata
meta = dataset.metadata
print("Dataset Name:", meta.name)
print("Description:", meta.description)
print("Spatial Coverage:", getattr(meta, 'spatialCoverage', None))
print("Date Published:", getattr(meta, 'datePublished', None))

## 2. Data Overview

List available record sets (by `@id`), and for each, review their fields (also by `@id`). This reveals the dataset's tabular organization.

In [ ]:
print("\nAvailable Record Sets (by @id):\n")
for rs in dataset.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")
    if 'field' in rs:
        print("  Fields:")
        for fld in rs['field']:
            # Each field is an object or an @id
            if isinstance(fld, dict):
                print(f"    - {fld['@id']}: {fld.get('name', '')}")
            else:
                print(f"    - {fld}")
    else:
        print("  [No fields registered]")

## 3. Data Extraction
Load data for each record set by referencing its `@id`, and display the columns for inspection.

Choose a target record set `@id` from the above listing for demonstration below.

In [ ]:
# List all record set @ids (for later use)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Found record sets:", record_set_ids)

# We'll use the first available record set as a demonstration:
if record_set_ids:
    selected_rs_id = record_set_ids[0]
    print(f"\nLoading records from record set: {selected_rs_id}")
else:
    raise Exception('No record sets found in the dataset!')

# Extract data for each record set into dataframes
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} records for record set {rs_id}.")

# Show example columns for the chosen record set
print('\nColumns for', selected_rs_id, ':')
print(list(dataframes[selected_rs_id].columns))
dataframes[selected_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing using field/column `@id`. If available, filter on a numeric column, normalize values, and optionally group by another column's `@id`.

In [ ]:
# Identify numeric fields in the selected record set
df = dataframes[selected_rs_id]
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print("Numeric fields available:", numeric_fields)

# Pick the first numeric field (or define by @id if known)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise Exception('No numeric fields found in the selected record set.')

threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records shown below.")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized values for {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a categorical/grouping field by @id, if available
categorical_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field_id = None
for fld in categorical_fields:
    # Skip all-nan or empty fields
    if not df[fld].isnull().all() and df[fld].nunique() < len(df) // 2:
        group_field_id = fld
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}, mean of {numeric_field_id}:")
    print(grouped_df.head())
else:
    print('No suitable categorical field found for grouping!')

## 5. Visualization

Plot the distribution of the (filtered and normalized) numeric field, and, if grouped in the EDA step, a barplot for group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the normalized numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True)
plt.title(f"Distribution of normalized {numeric_field_id} (filtered)")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Count")
plt.show()

# If grouped, plot barplot
if group_field_id and 'grouped_df' in locals():
    plt.figure(figsize=(8,4))
    sns.barplot(
        data=grouped_df, x=group_field_id, y=numeric_field_id
    )
    plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

Using the Croissant schema, we loaded FAIR² survey regression results and explored one of its record sets. This included metadata audit, tabular field extraction by `@id`, numeric filtering, normalization, grouping, and basic visualizations—all using `mlcroissant`'s transparent, schema-compliant API. Adapt this workflow to dig deeper into other record sets and analytical tasks in the dataset.